In [7]:
import cv2
import numpy as np
import math

# Load the video file
video_path = 'sequence_1.mp4'
cap = cv2.VideoCapture(video_path)

# Check if the video file is opened successfully
if not cap.isOpened():
    print("Error: Could not open video file.")
    exit()

# Read the first frame
ret, frame1 = cap.read()
gray1 = cv2.cvtColor(frame1, cv2.COLOR_BGR2GRAY)

# Parameters for Lucas-Kanade optical flow
lk_params = dict(winSize=(15, 15), criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 10, 0.03))

# Use goodFeaturesToTrack to find feature points in the first frame
p0 = cv2.goodFeaturesToTrack(gray1, maxCorners=0, qualityLevel=0.3, minDistance=7)

# Create a mask for drawing purposes
mask = np.zeros_like(frame1)

# Initialize total distance
total_distance = 0

average_vx_list = []
average_vy_list = []
V_list = []

while True:
    # Read the next frame
    ret, frame2 = cap.read()

    # Break the loop if no more frames are available
    if not ret:
        break

    # Convert the frame to grayscale
    gray2 = cv2.cvtColor(frame2, cv2.COLOR_BGR2GRAY)

    # Calculate optical flow from the previous frame to the current frame
    p1, st, err = cv2.calcOpticalFlowPyrLK(gray1, gray2, p0, None, **lk_params)

    # Select good points
    good_new = p1[st == 1]
    good_old = p0[st == 1]


    step = 65.5
    dt = step/1000 # dt in second fixed for all frames

    vx_list = []
    vy_list = []

    # Draw the tracks and calculate distances
    for i, (new, old) in enumerate(zip(good_new, good_old)):
        a, b = np.int32(new.ravel())
        c, d = np.int32(old.ravel())

        # calculate the speed in x and y axis
        vx = abs((a-c)/dt)
        vy = abs((b-d)/dt)
        vx_list.append(vx)
        vy_list.append(vy)


        mask = cv2.line(mask, (a, b), (c, d), (0, 255, 0), 2)
        frame2 = cv2.circle(frame2, (a, b), 5, (0, 0, 255), -1)


    # calculate the average speed in x and y axis
    average_vx = sum(vx_list) / len(vx_list)
    average_vy = sum(vy_list) / len(vy_list)    
    V = math.sqrt(pow(average_vx,2) + pow(average_vy,2))
    
    average_vx_list.append(average_vx)
    average_vy_list.append(average_vy)
    V_list.append(V)

    # Display the result
    result = cv2.add(frame2, mask)
    cv2.imshow('Optical Flow', result)

    # Update the previous frame and points
    gray1 = gray2.copy()
    p0 = good_new.reshape(-1, 1, 2)

    # Break the loop if 'q' key is pressed
    if cv2.waitKey(30) & 0xFF == ord('q'):
        break

# Release the video capture object
cap.release()
cv2.destroyAllWindows()


In [8]:
import pandas as pd
# Create a DataFrame
df = pd.DataFrame({'vx': average_vx_list, 'vy': average_vy_list, 'v': V_list})
df = df.head(34)

In [9]:
df.head()

,vx,vy,v
0,28.649515,336.443314,337.660922
1,15.832627,272.735840,273.195005
2,50.381679,345.801527,349.452442
3,33.778626,238.549618,240.929276
4,51.145038,330.152672,334.090709


In [10]:
# Save the DataFrame to a CSV file
df.to_csv('Predicted_velocity/Lucas-Kanade/Shi-Tomasi.csv', index=False)

# FAST (Features from Accelerated Segment Test)

In [11]:
# Load the video file
video_path = 'sequence_1.mp4'
cap = cv2.VideoCapture(video_path)

# Check if the video file is opened successfully
if not cap.isOpened():
    print("Error: Could not open video file.")
    exit()

# Read the first frame
ret, frame1 = cap.read()
gray1 = cv2.cvtColor(frame1, cv2.COLOR_BGR2GRAY)

# Parameters for Lucas-Kanade optical flow
lk_params = dict(winSize=(15, 15), criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 10, 0.03))

# Use FAST to find feature points in the first frame
fast = cv2.FastFeatureDetector_create(threshold=20, nonmaxSuppression=True)
kp = fast.detect(gray1, None)
p0 = np.float32([kp[i].pt for i in range(len(kp))]).reshape(-1, 1, 2)

# Create a mask for drawing purposes
mask = np.zeros_like(frame1)

# Initialize total distance
total_distance = 0

average_vx_list = []
average_vy_list = []
V_list = []

while True:
    # Read the next frame
    ret, frame2 = cap.read()

    # Break the loop if no more frames are available
    if not ret:
        break

    # Convert the frame to grayscale
    gray2 = cv2.cvtColor(frame2, cv2.COLOR_BGR2GRAY)

    # Calculate optical flow from the previous frame to the current frame
    p1, st, err = cv2.calcOpticalFlowPyrLK(gray1, gray2, p0, None, **lk_params)

    # Select good points
    good_new = p1[st == 1]
    good_old = p0[st == 1]

    step = 65.5
    dt = step / 1000  # dt in seconds fixed for all frames

    vx_list = []
    vy_list = []

    # Draw the tracks and calculate distances
    for i, (new, old) in enumerate(zip(good_new, good_old)):
        a, b = np.int32(new.ravel())
        c, d = np.int32(old.ravel())

        # calculate the speed in x and y axis
        vx = abs((a - c) / dt)
        vy = abs((b - d) / dt)
        vx_list.append(vx)
        vy_list.append(vy)

        mask = cv2.line(mask, (a, b), (c, d), (0, 255, 0), 2)
        frame2 = cv2.circle(frame2, (a, b), 5, (0, 0, 255), -1)

    # calculate the average speed in x and y axis
    average_vx = sum(vx_list) / len(vx_list)
    average_vy = sum(vy_list) / len(vy_list)
    V = math.sqrt(pow(average_vx, 2) + pow(average_vy, 2))

    average_vx_list.append(average_vx)
    average_vy_list.append(average_vy)
    V_list.append(V)

    # Display the result
    result = cv2.add(frame2, mask)
    cv2.imshow('Optical Flow', result)

    # Update the previous frame and points
    gray1 = gray2.copy()
    
    # Use FAST to find feature points in the current frame
    kp = fast.detect(gray1, None)
    p0 = np.float32([kp[i].pt for i in range(len(kp))]).reshape(-1, 1, 2)

    # Break the loop if 'q' key is pressed
    if cv2.waitKey(30) & 0xFF == ord('q'):
        break

# Release the video capture object
cap.release()
cv2.destroyAllWindows()


In [12]:
# Create a DataFrame
df_fast = pd.DataFrame({'vx': average_vx_list, 'vy': average_vy_list, 'v': V_list})
df_fast = df_fast.head(34)
df_fast.head()

,vx,vy,v
0,27.896478,352.971645,354.072303
1,16.039527,280.825976,281.283656
2,35.982029,326.609971,328.586031
3,28.101100,269.067307,270.530752
4,37.145721,307.263603,309.500770


In [13]:
# Save the DataFrame to a CSV file
df_fast.to_csv('Predicted_velocity/Lucas-Kanade/FAST.csv', index=False)